# Project-H: Grafting Action Heads onto Frozen VLMs

This notebook walks through the core idea of **project-h**: take a frozen Vision-Language Model (SmolVLM-256M-Instruct), graft a tiny task-specific action head onto it, and train that head to control an agent in a visual navigation task.

By the end you will have:
- A trained `JoystickAppendage` wrapped in a `VisionBridge` skip connection
- Benchmark numbers comparing the graft to an analytical expert
- Side-by-side GIFs of expert vs trained agent

## 1. Install

In [ ]:
!pip install -q git+https://github.com/jerod92/project-h.git@claude/vla-robotic-hands-platform-kGiza

## 2. Device Setup

In [ ]:
import torch

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(f"Using device: {device}")

## 3. The Environment

`TargetNavEnvironment` renders a 224x224 image. The agent (red circle) must navigate to the target (blue X) using continuous `(dx, dy)` joystick actions. Episodes end on contact or after `max_steps=80` steps.

We first visualise the initial observation, then run the **analytical expert** to see what optimal behaviour looks like and establish a baseline success rate.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from project_h import TargetNavEnvironment, run_expert_baseline

env = TargetNavEnvironment(width=224, height=224, max_steps=80)
obs = env.reset(seed=42)

plt.figure(figsize=(3, 3))
plt.imshow(obs)
plt.axis("off")
plt.title("Initial observation")
plt.tight_layout()
plt.show()

print(f"Observation shape : {np.array(obs).shape}")
print(f"Action space      : continuous (dx, dy) in [-1, 1]")

In [ ]:
# Show 6 frames of an expert rollout
env_viz = TargetNavEnvironment(width=224, height=224, max_steps=80)
obs = env_viz.reset(seed=0)

frames = [obs]
done = False
while not done and len(frames) < 6:
    action = env_viz.expert_action()
    _step = env_viz.step(action)
    obs, reward, done, info = _step.observation, _step.reward, _step.done, _step.info
    frames.append(obs)

fig, axes = plt.subplots(1, len(frames), figsize=(3 * len(frames), 3))
for i, (ax, frame) in enumerate(zip(axes, frames)):
    ax.imshow(frame)
    ax.axis("off")
    ax.set_title(f"Step {i}")
plt.suptitle("Expert rollout (first 6 frames)", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Expert baseline success rate
expert_stats = run_expert_baseline(env, n_episodes=10)
expert_success = expert_stats["success_rate"]
print(f"Expert baseline success rate: {expert_success:.1%}")

## 4. Load SmolVLM-256M-Instruct

We load the VLM in float16 to save memory, then freeze all its weights. Only the action head will be trained.

In [ ]:
from transformers import AutoProcessor, AutoModelForVision2Seq
from project_h import VLAGraft

model_id = "HuggingFaceTB/SmolVLM-256M-Instruct"

processor = AutoProcessor.from_pretrained(model_id)
vlm = AutoModelForVision2Seq.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
).to(device)

# Freeze the VLM — we never update its weights
for param in vlm.parameters():
    param.requires_grad = False

n_params   = sum(p.numel() for p in vlm.parameters()) / 1e6
hidden_dim = vlm.config.hidden_size
vision_dim = VLAGraft.detect_vision_dim(vlm)

print(f"VLM parameters : {n_params:.0f}M (frozen)")
print(f"LLM hidden dim : {hidden_dim}")
print(f"Vision enc dim : {vision_dim}")

## 5. Build the Graft

Three components are assembled:

1. **`JoystickAppendage`** — a small MLP that predicts continuous `(dx, dy)` from LLM hidden states.
2. **`VisionBridge`** — wraps the appendage with a **skip connection** from the VLM's vision encoder.
3. **`VLAGraft`** — pairs the frozen VLM with the wrapped appendage.

### Why VisionBridge matters

Without a skip connection the spatial image features must travel through the entire LLM (256M parameters, many layers of attention) before reaching the action head. This creates a very long credit-assignment path and the action head struggles to pick up spatial signals during training.

`VisionBridge` short-circuits this: it reads the vision encoder's output directly and injects those spatial features alongside the LLM hidden states. In practice this changes convergence from **~30% success after BC** to **~100%**. Always use it for image-based tasks.

In [ ]:
from project_h import (
    JoystickAppendage,
    VisionBridge,
    VLAGraft,
    GraftConfig,
)

appendage = JoystickAppendage(hidden_dim=256)
bridged   = VisionBridge(appendage=appendage, vision_dim=vision_dim)
graft     = VLAGraft(vlm=vlm, appendage=bridged, config=GraftConfig())

trainable = sum(p.numel() for p in graft.parameters() if p.requires_grad)
print(f"Trainable parameters : {trainable:,}  ({trainable / 1e6:.3f}M)")
print(f"Frozen VLM           : {n_params:.0f}M")
print(f"Train/total ratio    : {trainable / (n_params * 1e6 + trainable):.2%}")

## 6. Train

Training uses a two-phase **curriculum**:

- **Behavioural Cloning (BC)** — supervised imitation of the analytical expert. Runs until loss < `bc_early_stop_loss` or `bc_steps` steps are exhausted.
- **Reinforcement Learning (RL)** — online fine-tuning with a sparse reward (+1 on success). Runs until batch success >= `rl_early_stop_success` or `rl_steps` steps are exhausted.

Only the action head weights are updated; the VLM stays frozen throughout.

In [ ]:
import os
from project_h import CurriculumConfig, TrainingCurriculum, QUICK_CURRICULUM

os.makedirs("model_checkpoints/intro_joystick", exist_ok=True)

curriculum_config = CurriculumConfig(
    bc_steps=500,
    rl_steps=100,
    bc_early_stop_loss=0.03,
    rl_early_stop_success=0.9,
    appendage_lr=3e-4,
    device=device,
    save_dir="model_checkpoints/intro_joystick",
    freezing_stages=QUICK_CURRICULUM,
    eval_every=100,
    log_every=25,
    eval_episodes=5,
)

curriculum = TrainingCurriculum(
    graft=graft,
    processor=processor,
    env=env,
    config=curriculum_config,
)

history = curriculum.run()
print("Training complete.")

## 7. Training Curves

In [ ]:
bc_log = history.get("bc", {})
steps  = bc_log.get("steps", [])
losses = bc_log.get("loss",  [])

if steps and losses:
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.plot(steps, losses, linewidth=1.5, color="steelblue")
    ax.set_xlabel("BC step")
    ax.set_ylabel("Loss")
    ax.set_title("Behavioural Cloning — training loss")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("No BC loss history available (history keys:", list(history.keys()), ")")

## 8. Evaluate

`BenchmarkSuite` rolls out the trained graft for `n_episodes` episodes and reports success rate and mean episode reward. We compare to the analytical expert baseline recorded in section 3.

In [ ]:
from project_h import BenchmarkSuite

suite   = BenchmarkSuite(graft=graft, processor=processor, envs=[env], device=device)
results = suite.run_benchmark(env=env, n_episodes=20)

graft_success = results["success_rate"]
graft_reward  = results["mean_reward"]

print(f"Trained graft   — success rate: {graft_success:.1%}  mean reward: {graft_reward:.3f}")
print(f"Expert baseline — success rate: {expert_success:.1%}")
print(f"Gap to expert   : {expert_success - graft_success:+.1%}")

## 9. Save & Reload

The graft can be saved to disk and reloaded independently of the base VLM weights.

In [ ]:
save_path = "model_checkpoints/intro_joystick/final"
graft.save(save_path)
print(f"Saved to: {save_path}")

# Reload — pass the already-loaded frozen VLM to avoid re-downloading
graft_reloaded = VLAGraft.from_pretrained(
    save_path,
    vlm=vlm,
    device=device,
)
print("Reloaded successfully.")

# Sanity check — run one episode with the reloaded model
obs  = env.reset(seed=99)
done = False
total_reward = 0.0
while not done:
    action = graft_reloaded.predict(obs, processor=processor)
    _step = env.step(action)
    obs, reward, done, info = _step.observation, _step.reward, _step.done, _step.info
    total_reward += reward
print(f"Sanity-check episode — reward: {total_reward:.3f}  success: {info.get('success', False)}")

## 10. Export GIFs — Expert vs Trained

We record a GIF of the analytical expert and a GIF of the trained graft, then display them side by side.

In [ ]:
import os
from project_h import record_expert_gif, save_rollout_gif

os.makedirs("gifs", exist_ok=True)

record_expert_gif(
    env=env,
    path="gifs/expert.gif",
    n_steps=80,
    seed=7,
    fps=10,
)
print("Saved: gifs/expert.gif")

save_rollout_gif(
    graft=graft,
    processor=processor,
    env=env,
    path="gifs/trained.gif",
    n_steps=80,
    seed=7,
    device=device,
    fps=10,
)
print("Saved: gifs/trained.gif")

In [ ]:
from IPython.display import display, HTML

display(HTML("""
<table>
  <tr>
    <th style='text-align:center;padding:8px'>Expert</th>
    <th style='text-align:center;padding:8px'>Trained Graft</th>
  </tr>
  <tr>
    <td><img src='gifs/expert.gif' width='200'/></td>
    <td><img src='gifs/trained.gif' width='200'/></td>
  </tr>
</table>
"""))